In [1]:
# complited embedding,vectore store,semantic retrieval,build prompt with retrieved context and generated the answer
import langchain
print("1.done")
import langchain_community
print("2.done")
from langchain_community.document_loaders import TextLoader
print("3.done")
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("4.done")
import faiss
print("5.done")
from sentence_transformers import SentenceTransformer
print("6.done")

1.done
2.done
3.done
4.done
5.done
6.done


In [2]:
loader = TextLoader("sample.txt")
documents = loader.load()
print(documents)

[Document(metadata={'source': 'sample.txt'}, page_content='finally i have completed the importing files it took me so much time still i am happy i learned so many things while dealing with the errors and focusing on my 1st project agentic rag optimizer and i wish i could complete the project without any more conflicts it is asking me to write some more words that it can work perfectly now it is showin chunk 1 created because i have created the smaller text and i give  chunk size 100 so ihave to create more thank 100 so then it will create more if i will increase the text size thats what i am doing right now i wish it have 100 chunks.\nit has several sentences to test the test splitter functionality.\nhere is a second paragraph to ensure we get multiple chunks\nfinally the third paragraph will help check overlaping behaviour')]


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50,
                                               chunk_overlap=5)
docs = text_splitter.split_documents(documents)
print("number of chunks:", len(docs))
print(docs[0].page_content)

number of chunks: 20
finally i have completed the importing files it


In [4]:
#embeddings
model = SentenceTransformer("all-miniLM-L6-v2")
texts = [doc.page_content for doc in docs]
embeddings = model.encode(texts)
print("no. of embeddings:", len(embeddings))
print("vector dimension :", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\RADHA\anaconda3\envs\rag_env\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RADHA\.cache\huggingface\hub\models--sentence-transformers--all-miniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

no. of embeddings: 20
vector dimension : 384


In [5]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("total vectors stored:", index.ntotal)

total vectors stored: 20


In [21]:
query ="which optimizer ?"

In [22]:
query_vector = model.encode([query])

In [23]:
k = 3
distance,indices = index.search(query_vector, k)
print(indices)

[[ 4 12 15]]


In [24]:
for i in indices[0]:
    print(docs[i].page_content)
    print("-----")

rag optimizer and i wish i could complete the
-----
what i am doing right now i wish it have 100
-----
test splitter functionality.
-----


In [25]:
context = ""

for i in indices[0]:
    context += docs[i].page_content + "\n"
print(context)

rag optimizer and i wish i could complete the
what i am doing right now i wish it have 100
test splitter functionality.



In [26]:
# building the prompt
prompt = f""" Answer the question using the context below

context: {context}

question: {query}

Answer: """

print(prompt)

 Answer the question using the context below

context: rag optimizer and i wish i could complete the
what i am doing right now i wish it have 100
test splitter functionality.


question: which optimizer ?

Answer: 


In [27]:
from transformers import pipeline

In [30]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

In [31]:
response = generator(prompt, max_length=200)
print(response[0]["generated_text"])

rag optimizer
